# 01 · Question 1 — Is the backtest real, or overfit?

Every candidate is run through the same top-50, equal-weight, 10 bps weekly backtest
(`sv/backtest.py`) and judged on **active return vs the equal-weighted tradable universe**, which cancels
the survivorship and universe effects that all candidates share.

Three tests, three failure modes (`sv/validation/overfit.py`):

| test | guards against | pass rule |
|---|---|---|
| Stationary block-bootstrap CI on annualised Sharpe | trusting a point estimate | 95% CI excludes 0 |
| Deflated Sharpe Ratio (Bailey & López de Prado) | picking the best of N tries | DSR ≥ 0.95 |
| Cross-sectional permutation null | signal–return link being coincidental | p < 0.05 |

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from sv import db

con = db.connect(read_only=True)
q   = lambda name, **p: db.run_sql_file(con, name, p or None)   # run sql/queries/<name>.sql
sql = lambda text, **p: db.read(con, text, p or None)                # run an inline query
pd.set_option("display.width", 140); plt.rcParams["figure.figsize"] = (10, 4)

## Backtest summary (full sample)

In [ ]:
q("champion_challenger", start=config.BACKTEST_START).round(3)

## Validation results

In [ ]:
vr = sql("""SELECT model, test, ROUND(statistic,3) AS statistic, ROUND(ci_low,3) AS ci_low, ROUND(ci_high,3) AS ci_high,
                   ROUND(p_value,3) AS p_value, verdict, detail FROM validation_results ORDER BY model, test""")
vr

In [ ]:
vr.pivot(index="model", columns="test", values="verdict")

## Bootstrap distribution of the Sharpe ratio

The point estimate is the vertical line; the histogram is what the same strategy could have produced
under resampled (block-preserving) histories.

In [ ]:
from sv.validation import overfit
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, m in zip(axes, ["momentum", "gbm_expected", "gbm"]):
    r = overfit.active_returns(con, m)
    rng = np.random.default_rng(0)
    boots = [overfit.sharpe(r.values[overfit.stationary_bootstrap_indices(len(r), config.BOOTSTRAP_BLOCK, rng)]) for _ in range(2000)]
    ax.hist(boots, bins=40, alpha=.7); ax.axvline(overfit.sharpe(r), color="k"); ax.axvline(0, color="r", ls="--")
    ax.set_title(f"{m}: SR={overfit.sharpe(r):.2f}")
plt.show()

## Reading the DSR

With only 3 candidates the expected *maximum* Sharpe of pure-noise strategies is already positive.
A model must beat that hurdle, not zero. This is the trading analogue of peeking in sequential A/B
testing: the more you look, the higher the bar.

In [ ]:
vr[vr.test == "deflated_sharpe"][["model", "statistic", "verdict", "detail"]]